In [12]:
# Import libraries
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context='notebook', style='whitegrid')

In [13]:
df_part = pd.read_csv('../data/polymod/participants.csv')
df_cnt = pd.read_csv('../data/polymod/contacts.csv')

In [14]:
# Filter out non-German participants
pid_min, pid_max = df_part['global_id'].min(), df_part['global_id'].max()
df_cnt = df_cnt[(df_cnt['part_id'] >= pid_min) & (df_cnt['part_id'] <= pid_max)]

In [15]:
df_cnt['age_cnt'] = np.where(df_cnt['cnt_age_exact'].isna(),
                             (df_cnt['cnt_age_est_min'] + df_cnt['cnt_age_est_max']) // 2,
                             df_cnt['cnt_age_exact'])
df_cnt.dropna(subset='age_cnt', inplace=True)
df_cnt['age_cnt'] = df_cnt['age_cnt'].astype(int)

In [16]:
# Standardize column names
df_cnt.rename(columns={'cnt_gender': 'sex_cnt'}, inplace=True)
df_part.rename(columns={'participant_age': 'age_part', 'participant_gender': 'sex_part'}, inplace=True)

In [17]:
# Count the number of participants by age
df_n = (df_part.dropna(subset='age_part')
        .groupby(['age_part', 'sex_part'])
        .size()
        .reset_index(name='n'))

In [18]:
# Group contacts
df_part['class_size'] = df_part['class_size'].fillna(0)
df_part['work_contacts_nr'] = df_part['work_contacts_nr'].fillna(0)
df_part['y_grp'] = df_part['class_size'] + df_part['work_contacts_nr']
df_part['y_grp'] = df_part['y_grp'].apply(lambda x: min(x, 60))
df_grp = df_part.groupby('age_part').agg({'y_grp': 'sum'}).reset_index()

In [19]:
# Combine participant information and contact information
df_merged = pd.merge(df_cnt, df_part,
                     how='left',
                     left_on='part_id',
                     right_on='global_id')

# Count the number of contacts by age of participant and contact
df_sum = df_merged.groupby(['age_part', 'sex_part', 'age_cnt', 'sex_cnt']).size().reset_index(name='y')
print(df_sum.shape)

(3549, 3)


In [20]:
# Create a grid
def expand_grid(data_dict):
  rows = itertools.product(*data_dict.values())
  return pd.DataFrame.from_records(rows, columns=data_dict.keys())

df_grid = expand_grid({
  'age_part': np.arange(0, 85, 1),
  'sex_part': ['M', 'F'],
  'age_cnt': np.arange(0, 85, 1),
  'sex_cnt': ['M', 'F'],
})

df_sum = pd.merge(df_grid, df_sum, how='left', on=['age_part', 'sex_part', 'age_cnt', 'sex_cnt'])
df_sum = pd.merge(df_sum, df_n, how='left', on=['age_part', 'sex_part'])
df_sum['y'] = np.where(df_sum['y'].isna() & ~df_sum['n'].isna(), 0, df_sum['y'])
df_sum = df_sum.dropna(subset=['y'])

print(df_sum.shape)

(7055, 4)


In [21]:
df_sum = pd.merge(df_sum, df_grp, how='left', on='age_part')
df_sum['y_grp'] = df_sum['y_grp'].fillna(0)
df_sum['s'] = np.where(df_sum['y'] == 0, 1,
              np.where(df_sum['y'] + df_sum['y_grp'] > 0, df_sum['y'] / (df_sum['y'] + df_sum['y_grp']), 1))